# Week 5: IDS and one-page Nôm recognition preparation

This notebook uses the pinned course dictionaries. It demonstrates **text-level** IDS tokenization; it does not segment radical regions from pixels. Run the resource download command in the handout before starting.

## 1. IDS as a prefix expression

An IDC comes before its operands: `⿰日月` means 日 is left of 月. Binary IDCs such as `⿰`, `⿱`, `⿴`, and `⿻` take two operands; `⿲` and `⿳` take three. A direct CJKVI IDS describes one layer. The recognizer's expanded IDS recursively replaces components until it reaches its terminal vocabulary.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

from nom_ocr_lab.annotations import read_line_annotations
from nom_ocr_lab.recognition import (
    load_cjkvi_ids,
    load_expanded_ids,
    load_vocabulary,
    reverse_ids,
    text_to_ids,
)

workspace_value = os.environ.get("LAB_WORKSPACE")
if not workspace_value:
    raise RuntimeError("Set LAB_WORKSPACE before starting Jupyter")
workspace = Path(workspace_value)
ids_root = workspace / "datasets" / "ids"
sample_root = workspace / "datasets" / "nomnaocr-sample"
print(f"Lab workspace: {workspace}")

In [ ]:
expanded = load_expanded_ids(ids_root / "nom-ids" / "ids_exp.txt")
vocabulary = load_vocabulary(ids_root / "nom-ids" / "vocab_ids.txt")
ids_to_characters = reverse_ids(expanded)
direct = load_cjkvi_ids(ids_root / "cjkvi" / "ids.txt")
direct.update(load_cjkvi_ids(ids_root / "cjkvi" / "ids-ext-cdef.txt"))
print(f"Expanded character entries: {len(expanded):,}")
print(f"Recognizer tokens, in checkpoint order: {len(vocabulary):,}")
print(f"Direct CJKVI entries: {len(direct):,}")

## 2. Runnable reference converter and exact Chinese check

`text_to_ids` emits one structured record per non-whitespace Unicode code point. Copying the whole string into one field would lose the character boundaries needed later. Reimplement this behavior in your team repository.

In [ ]:
expected = {
    "休": "⿰亻木",
    "好": "⿰女子",
    "字": "⿱宀子",
    "明": "⿰日月",
    "林": "⿰木木",
}
reference = text_to_ids("休好字明林", expanded)
observed = {record.character: record.ids for record in reference}
assert observed == expected, (observed, expected)
for record in reference:
    candidates = ids_to_characters[record.ids]
    print(record.codepoint, record.character, direct.get(record.character), record.ids, list(record.tokens), candidates)
print("Five Chinese reference conversions passed.")

Before continuing, write `明`, `休`, and one nested character as IDS by hand. Compare the spatial relation you chose with the direct column printed above, then explain why a direct and an expanded IDS can differ.

## 3. Choose one durable Week 1 Nôm page

Set `WEEK05_PAGE` to a path relative to `nomnaocr-sample/images` if you want a page other than the first sorted image. The same relative path locates its line-label file.

In [ ]:
image_root = sample_root / "images"
image_suffixes = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
available = sorted(
    path.relative_to(image_root)
    for path in image_root.rglob("*")
    if path.is_file() and path.suffix.lower() in image_suffixes
)
if not available:
    raise RuntimeError(f"No Week 1 sample images found below {image_root}")
requested = os.environ.get("WEEK05_PAGE")
relative_image = Path(requested) if requested else available[0]
if relative_image not in available:
    raise RuntimeError(f"WEEK05_PAGE is not in the durable sample: {relative_image}")
image_path = image_root / relative_image
line_path = sample_root / "line_labels" / relative_image.with_suffix(".txt")
annotations = read_line_annotations(line_path)
print(f"Selected page: {relative_image}")
print(f"Line transcriptions: {len(annotations)}")

In [ ]:
page_records = []
line_summary = []
for line_number, annotation in enumerate(annotations, 1):
    records = text_to_ids(annotation.text, expanded)
    page_records.extend(records)
    covered = sum(record.status == "covered" for record in records)
    total = len(records)
    line_summary.append((line_number, annotation.text, covered, total, total - covered))

print("line | covered/total | OOV | transcription")
for line_number, text, covered, total, oov in line_summary:
    print(f"{line_number:>4} | {covered:>3}/{total:<3} | {oov:>3} | {text}")
covered_total = sum(record.status == "covered" for record in page_records)
page_total = len(page_records)
coverage = covered_total / page_total if page_total else 0.0
print(f"Page lookup coverage: {covered_total}/{page_total} = {coverage:.2%}")
oov_counts = Counter(record.character for record in page_records if record.status == "oov")
print("OOV counts:", dict(oov_counts))

### zi.tools verification worksheet

Choose at least five **covered** Nôm characters from the output. Search each at [zi.tools](https://zi.tools/) and complete this table. Do not treat a variant or direct/expanded difference as an automatic error.

| Character | Unicode | Course expanded IDS | zi.tools IDS/description | Same, raw/expanded, or glyph variant? |
| --- | --- | --- | --- | --- |
| … | … | … | … | … |
| … | … | … | … | … |
| … | … | … | … | … |
| … | … | … | … | … |
| … | … | … | … | … |

## 4. Inspect exported detector crops

Run the detector and crop-export commands from the handout first. These boxes remain unlabeled: line transcription order is not a safe box-to-character assignment.

In [ ]:
default_crop_root = workspace / "datasets" / "nomnaocr-recognition-crops" / image_path.stem
crop_root = Path(os.environ.get("WEEK05_CROP_ROOT", default_crop_root))
manifest_path = crop_root / "manifest.json"
if not manifest_path.is_file():
    print(f"No crop manifest yet: {manifest_path}")
    print("Run the detector and export_recognition_crops commands in the handout, then rerun this cell.")
else:
    crop_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    entries = crop_manifest["entries"][:24]
    columns = 6
    rows = max(1, (len(entries) + columns - 1) // columns)
    figure, axes = plt.subplots(rows, columns, figsize=(12, 2 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, entry in zip(axes.flat, entries):
        with Image.open(crop_root / entry["crop"]) as crop:
            axis.imshow(crop.convert("RGB"))
        confidence = entry["confidence"]
        suffix = "" if confidence is None else f"  conf={confidence:.2f}"
        axis.set_title(f"box {entry['box_id']}{suffix}", fontsize=8)
        axis.axis("off")
    figure.suptitle("Unlabeled recognizer inference crops")
    figure.tight_layout()
    plt.show()
    print(f"Displayed {len(entries)} of {crop_manifest['count']} crops.")

## Submission reflection

1. What information does `⿰` add that an unordered radical set would lose?
2. Why must `vocab_ids.txt` retain its original order?
3. Which page characters were OOV, and were they uncertain marks, punctuation, or ideographs?
4. What direct/expanded or glyph-region differences appeared in zi.tools?
5. Why would pairing the first detected box with the first transcription character create false ground truth?